In [51]:
!pip install pyspark -q

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Modelo_Predictivo_Ventas_MLlib")
    .getOrCreate()
)

print("SPARK INICIADO CORRECTAMENTE")
print("Versión:", spark.version)
print("Master:", spark.sparkContext.master)

SPARK INICIADO CORRECTAMENTE
Versión: 4.0.4
Master: local[*]


In [52]:
import pandas as pd

ruta_excel = "/content/03. Apoyo prueba - ventas_simuladas.xlsx"

df_pandas = pd.read_excel(
    ruta_excel,
    sheet_name="ventas_simuladas.csv"
)

ruta_csv = "/content/ventas_simuladas_prueba.csv"

df_pandas.to_csv(
    ruta_csv,
    index=False
)

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ruta_csv)
)

print("PRIMERAS FILAS DEL DATASET")
df.show(5, truncate=False)

print("\nESTRUCTURA DEL DATASET")
df.printSchema()

print("\nDIMENSIONES")
print("Filas:", df.count())
print("Columnas:", len(df.columns))

PRIMERAS FILAS DEL DATASET
+----------+--------+--------+---------------+-------------------+-------------------+
|Sucursal  |Producto|Cantidad|Precio_Unitario|Monto_Total        |Fecha_Hora         |
+----------+--------+--------+---------------+-------------------+-------------------+
|Sucursal A|Pan     |3.0     |989.78         |2969.34            |2023-01-04 17:17:12|
|Sucursal A|Aceite  |4.0     |563.57         |2254.28            |2023-01-03 14:34:53|
|Sucursal B|Galletas|5.0     |1339.04        |6695-02-01 00:00:00|invalid-date       |
|Sucursal B|Galletas|4.0     |1180.5         |4722.0             |2023-01-05 02:34:24|
|Sucursal D|Pan     |3.0     |2194.99        |6584.97            |2023-01-07 18:05:40|
+----------+--------+--------+---------------+-------------------+-------------------+
only showing top 5 rows

ESTRUCTURA DEL DATASET
root
 |-- Sucursal: string (nullable = true)
 |-- Producto: string (nullable = true)
 |-- Cantidad: double (nullable = true)
 |-- Precio_Unita

In [53]:
from pyspark.sql.functions import col, sum as spark_sum

print("VALORES NULOS POR COLUMNA")

df.select(
    [
        spark_sum(
            col(c).isNull().cast("int")
        ).alias(c)
        for c in df.columns
    ]
).show()

VALORES NULOS POR COLUMNA
+--------+--------+--------+---------------+-----------+----------+
|Sucursal|Producto|Cantidad|Precio_Unitario|Monto_Total|Fecha_Hora|
+--------+--------+--------+---------------+-----------+----------+
|       0|       6|      12|              0|          0|         0|
+--------+--------+--------+---------------+-----------+----------+



In [54]:
from pyspark.sql.functions import col, expr, hour

df_limpio = (
    df
    .withColumn(
        "Cantidad",
        expr("try_cast(Cantidad AS INT)")
    )
    .withColumn(
        "Precio_Unitario",
        expr("try_cast(Precio_Unitario AS DOUBLE)")
    )
    .withColumn(
        "Monto_Total",
        expr("try_cast(Monto_Total AS DOUBLE)")
    )
    .withColumn(
        "Fecha_Hora",
        expr(
            "try_to_timestamp("
            "Fecha_Hora, 'yyyy-MM-dd HH:mm:ss'"
            ")"
        )
    )
)

df_limpio = df_limpio.dropna(
    subset=[
        "Sucursal",
        "Producto",
        "Cantidad",
        "Precio_Unitario",
        "Monto_Total",
        "Fecha_Hora"
    ]
)

df_limpio = df_limpio.withColumn(
    "Hora",
    hour(col("Fecha_Hora"))
)

print("DATASET LIMPIO")
df_limpio.show(5, truncate=False)

print("\nESTRUCTURA DEL DATASET LIMPIO")
df_limpio.printSchema()

print("\nTOTAL DE REGISTROS VÁLIDOS")
print(df_limpio.count())

DATASET LIMPIO
+----------+--------+--------+---------------+-----------+-------------------+----+
|Sucursal  |Producto|Cantidad|Precio_Unitario|Monto_Total|Fecha_Hora         |Hora|
+----------+--------+--------+---------------+-----------+-------------------+----+
|Sucursal A|Pan     |3       |989.78         |2969.34    |2023-01-04 17:17:12|17  |
|Sucursal A|Aceite  |4       |563.57         |2254.28    |2023-01-03 14:34:53|14  |
|Sucursal B|Galletas|4       |1180.5         |4722.0     |2023-01-05 02:34:24|2   |
|Sucursal D|Pan     |3       |2194.99        |6584.97    |2023-01-07 18:05:40|18  |
|Sucursal B|Atún    |3       |2470.44        |7411.32    |2023-01-06 23:36:37|23  |
+----------+--------+--------+---------------+-----------+-------------------+----+
only showing top 5 rows

ESTRUCTURA DEL DATASET LIMPIO
root
 |-- Sucursal: string (nullable = true)
 |-- Producto: string (nullable = true)
 |-- Cantidad: integer (nullable = true)
 |-- Precio_Unitario: double (nullable = true)
 

In [55]:
from pyspark.sql.functions import when

df_modelo = df_limpio.withColumn(
    "label",
    when(
        (col("Monto_Total") > 7000)
        | (col("Hora") < 6),
        1.0
    ).otherwise(0.0)
)

print("DISTRIBUCIÓN DE LA VARIABLE LABEL")

df_modelo.groupBy(
    "label"
).count().orderBy(
    "label"
).show()

print("\nEJEMPLOS DE LA VARIABLE OBJETIVO")

df_modelo.select(
    "Sucursal",
    "Producto",
    "Cantidad",
    "Precio_Unitario",
    "Monto_Total",
    "Hora",
    "label"
).show(10, truncate=False)

DISTRIBUCIÓN DE LA VARIABLE LABEL
+-----+-----+
|label|count|
+-----+-----+
|  0.0|   79|
|  1.0|   53|
+-----+-----+


EJEMPLOS DE LA VARIABLE OBJETIVO
+----------+--------+--------+---------------+-----------+----+-----+
|Sucursal  |Producto|Cantidad|Precio_Unitario|Monto_Total|Hora|label|
+----------+--------+--------+---------------+-----------+----+-----+
|Sucursal A|Pan     |3       |989.78         |2969.34    |17  |0.0  |
|Sucursal A|Aceite  |4       |563.57         |2254.28    |14  |0.0  |
|Sucursal B|Galletas|4       |1180.5         |4722.0     |2   |1.0  |
|Sucursal D|Pan     |3       |2194.99        |6584.97    |18  |0.0  |
|Sucursal B|Atún    |3       |2470.44        |7411.32    |23  |1.0  |
|Sucursal C|Leche   |3       |1210.54        |3631.62    |19  |0.0  |
|Sucursal B|Aceite  |2       |826.8          |1653.6     |13  |0.0  |
|Sucursal C|Pan     |2       |2326.27        |4652.54    |2   |1.0  |
|Sucursal D|Galletas|4       |785.74         |3142.96    |17  |0.0  |
|Sucurs

In [56]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

indexador_sucursal = StringIndexer(
    inputCol="Sucursal",
    outputCol="Sucursal_index",
    handleInvalid="keep"
)

indexador_producto = StringIndexer(
    inputCol="Producto",
    outputCol="Producto_index",
    handleInvalid="keep"
)

assembler = VectorAssembler(
    inputCols=[
        "Sucursal_index",
        "Producto_index",
        "Cantidad",
        "Precio_Unitario",
        "Monto_Total",
        "Hora"
    ],
    outputCol="features",
    handleInvalid="skip"
)

print("VARIABLES PARA EL MODELO CONFIGURADAS")

VARIABLES PARA EL MODELO CONFIGURADAS


In [57]:
train, test = df_modelo.randomSplit(
    [0.80, 0.20],
    seed=42
)

train = train.cache()
test = test.cache()

n_train = train.count()
n_test = test.count()

print("DIVISIÓN DEL DATASET")
print("Entrenamiento:", n_train)
print("Prueba:", n_test)

print("\nDISTRIBUCIÓN DE LABEL EN ENTRENAMIENTO")

train.groupBy(
    "label"
).count().orderBy(
    "label"
).show()

print("DISTRIBUCIÓN DE LABEL EN PRUEBA")

test.groupBy(
    "label"
).count().orderBy(
    "label"
).show()

DIVISIÓN DEL DATASET
Entrenamiento: 108
Prueba: 24

DISTRIBUCIÓN DE LABEL EN ENTRENAMIENTO
+-----+-----+
|label|count|
+-----+-----+
|  0.0|   62|
|  1.0|   46|
+-----+-----+

DISTRIBUCIÓN DE LABEL EN PRUEBA
+-----+-----+
|label|count|
+-----+-----+
|  0.0|   17|
|  1.0|    7|
+-----+-----+



In [58]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0
)

pipeline = Pipeline(
    stages=[
        indexador_sucursal,
        indexador_producto,
        assembler,
        lr
    ]
)

print("PIPELINE CONFIGURADO CORRECTAMENTE")

PIPELINE CONFIGURADO CORRECTAMENTE


In [59]:
modelo_pipeline = pipeline.fit(train)

print("MODELO ENTRENADO CORRECTAMENTE")

modelo_lr = modelo_pipeline.stages[-1]

print(
    "Número de iteraciones:",
    modelo_lr.summary.totalIterations
)

MODELO ENTRENADO CORRECTAMENTE
Número de iteraciones: 14


In [60]:
predicciones = modelo_pipeline.transform(test)

print("PREDICCIONES DEL MODELO")

predicciones.select(
    "label",
    "prediction",
    "probability"
).show(
    20,
    truncate=False
)

PREDICCIONES DEL MODELO
+-----+----------+-----------------------------------------+
|label|prediction|probability                              |
+-----+----------+-----------------------------------------+
|0.0  |0.0       |[0.9996727932930393,3.272067069607365E-4]|
|0.0  |0.0       |[0.9625189947299669,0.03748100527003306] |
|0.0  |0.0       |[0.9302950738040183,0.06970492619598168] |
|0.0  |0.0       |[0.9235987771400399,0.0764012228599601]  |
|0.0  |0.0       |[0.9926826838668499,0.00731731613315012] |
|0.0  |0.0       |[0.9586368594381832,0.04136314056181678] |
|0.0  |0.0       |[0.9814946177020168,0.018505382297983153]|
|0.0  |0.0       |[0.9310847006565419,0.06891529934345808] |
|0.0  |0.0       |[0.573097623151276,0.426902376848724]    |
|0.0  |0.0       |[0.981742306193961,0.018257693806039033] |
|1.0  |0.0       |[0.8199347613425402,0.18006523865745983] |
|1.0  |1.0       |[0.254343372219226,0.745656627780774]    |
|0.0  |0.0       |[0.9615830934495694,0.03841690655043062] |


In [61]:
print("FEATURES Y LABEL UTILIZADAS POR EL MODELO")

predicciones.select(
    "features",
    "label"
).show(
    10,
    truncate=False
)

print("\nESTRUCTURA DE LOS DATOS PROCESADOS")

predicciones.select(
    "features",
    "label",
    "prediction",
    "probability"
).printSchema()

FEATURES Y LABEL UTILIZADAS POR EL MODELO
+----------------------------------+-----+
|features                          |label|
+----------------------------------+-----+
|[3.0,0.0,3.0,557.91,1673.73,23.0] |0.0  |
|[3.0,3.0,2.0,1965.88,3931.76,20.0]|0.0  |
|[3.0,3.0,4.0,987.73,3950.92,14.0] |0.0  |
|[3.0,4.0,2.0,2429.44,4858.88,22.0]|0.0  |
|[3.0,1.0,1.0,621.23,621.23,13.0]  |0.0  |
|[3.0,6.0,1.0,1317.04,1317.04,17.0]|0.0  |
|[3.0,2.0,5.0,926.31,4631.55,18.0] |0.0  |
|[2.0,3.0,3.0,815.49,2446.47,12.0] |0.0  |
|[2.0,1.0,1.0,2306.81,2306.81,6.0] |0.0  |
|[2.0,1.0,2.0,1121.5,2243.0,15.0]  |0.0  |
+----------------------------------+-----+
only showing top 10 rows

ESTRUCTURA DE LOS DATOS PROCESADOS
root
 |-- features: vector (nullable = true)
 |-- label: double (nullable = false)
 |-- prediction: double (nullable = false)
 |-- probability: vector (nullable = true)



In [62]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluador_accuracy = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluador_accuracy.evaluate(
    predicciones
)

print("ACCURACY DEL MODELO")
print(f"{accuracy:.4f}")
print(f"Accuracy porcentual: {accuracy * 100:.2f}%")

ACCURACY DEL MODELO
0.8333
Accuracy porcentual: 83.33%


In [63]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluador_auc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = evaluador_auc.evaluate(
    predicciones
)

print("AUC DEL MODELO")
print(f"{auc:.4f}")

AUC DEL MODELO
0.8908


In [64]:
evaluador_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

f1 = evaluador_f1.evaluate(
    predicciones
)

print("F1-SCORE DEL MODELO")
print(f"{f1:.4f}")

F1-SCORE DEL MODELO
0.8333


In [65]:
print("RESULTADOS DEL MODELO")
print(f"Accuracy: {accuracy:.4f}")
print(f"AUC: {auc:.4f}")
print(f"F1-score: {f1:.4f}")

RESULTADOS DEL MODELO
Accuracy: 0.8333
AUC: 0.8908
F1-score: 0.8333


In [66]:
matriz_confusion = (
    predicciones
    .groupBy(
        "label",
        "prediction"
    )
    .count()
    .orderBy(
        "label",
        "prediction"
    )
)

print("MATRIZ DE CONFUSIÓN")

matriz_confusion.show()

MATRIZ DE CONFUSIÓN
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|   15|
|  0.0|       1.0|    2|
|  1.0|       0.0|    2|
|  1.0|       1.0|    5|
+-----+----------+-----+



In [67]:
print("TRANSACCIONES PREDICHAS COMO RIESGOSAS")

predicciones.filter(
    col("prediction") == 1.0
).select(
    "Sucursal",
    "Producto",
    "Cantidad",
    "Precio_Unitario",
    "Monto_Total",
    "Hora",
    "label",
    "prediction",
    "probability"
).show(
    20,
    truncate=False
)

TRANSACCIONES PREDICHAS COMO RIESGOSAS
+----------+--------+--------+---------------+-----------+----+-----+----------+----------------------------------------+
|Sucursal  |Producto|Cantidad|Precio_Unitario|Monto_Total|Hora|label|prediction|probability                             |
+----------+--------+--------+---------------+-----------+----+-----+----------+----------------------------------------+
|Sucursal B|Galletas|4       |1180.5         |4722.0     |2   |1.0  |1.0       |[0.254343372219226,0.745656627780774]   |
|Sucursal B|Pan     |5       |1971.47        |9857.35    |19  |1.0  |1.0       |[0.4273026284542383,0.5726973715457617] |
|Sucursal C|Aceite  |5       |2326.97        |11634.85   |7   |1.0  |1.0       |[0.00630219417574442,0.9936978058242556]|
|Sucursal D|Aceite  |5       |1826.0         |9130.0     |8   |1.0  |1.0       |[0.08136951803604757,0.9186304819639525]|
|Sucursal D|Fideos  |3       |1840.97        |5522.91    |10  |0.0  |1.0       |[0.1344394347897227,0.86556

In [68]:
from pyspark.ml.functions import vector_to_array

predicciones_analisis = predicciones.withColumn(
    "Probabilidad_Riesgo",
    vector_to_array(
        col("probability")
    )[1]
)

print("TRANSACCIONES CON MAYOR PROBABILIDAD DE RIESGO")

predicciones_analisis.select(
    "Sucursal",
    "Producto",
    "Monto_Total",
    "Hora",
    "label",
    "prediction",
    "Probabilidad_Riesgo"
).orderBy(
    col("Probabilidad_Riesgo").desc()
).show(
    15,
    truncate=False
)

TRANSACCIONES CON MAYOR PROBABILIDAD DE RIESGO
+----------+--------+-----------+----+-----+----------+-------------------+
|Sucursal  |Producto|Monto_Total|Hora|label|prediction|Probabilidad_Riesgo|
+----------+--------+-----------+----+-----+----------+-------------------+
|Sucursal C|Aceite  |11634.85   |7   |1.0  |1.0       |0.9936978058242556 |
|Sucursal D|Aceite  |9130.0     |8   |1.0  |1.0       |0.9186304819639525 |
|Sucursal D|Fideos  |5522.91    |10  |0.0  |1.0       |0.8655605652102774 |
|Sucursal B|Galletas|4722.0     |2   |1.0  |1.0       |0.745656627780774  |
|Sucursal D|Pan     |4992.0     |6   |0.0  |1.0       |0.6096456056656894 |
|Sucursal B|Pan     |9857.35    |19  |1.0  |1.0       |0.5726973715457617 |
|Sucursal D|Galletas|2167.39    |5   |1.0  |1.0       |0.5297089370170547 |
|Sucursal B|Galletas|2306.81    |6   |0.0  |0.0       |0.426902376848724  |
|Sucursal D|Pan     |6584.97    |18  |0.0  |0.0       |0.3272586153038317 |
|Sucursal D|Galletas|2098.38    |5   |1.0

Evaluación y justificación del modelo

Para este problema seleccioné un modelo de regresión logística, ya que la variable objetivo corresponde a una clasificación binaria entre transacciones normales y riesgosas. Además, este algoritmo permite obtener tanto una predicción como una probabilidad asociada a cada clase, lo que puede ser útil para priorizar casos que requieran revisión.

La variable label fue construida utilizando una regla simple relacionada con el contexto del ejercicio. Se consideró riesgosa una transacción cuando su monto supera los $7.000 o cuando ocurre durante la madrugada. De esta manera, la etiqueta permite simular un escenario de clasificación supervisada utilizando las variables disponibles en el dataset.

Para preparar los datos se utilizaron StringIndexer para transformar las variables categóricas y VectorAssembler para reunir las variables predictoras en una única columna features. Posteriormente, el preprocesamiento y el modelo de regresión logística se integraron mediante un Pipeline, permitiendo mantener un flujo reproducible entre entrenamiento y predicción.

El conjunto de datos se dividió en entrenamiento y prueba utilizando randomSplit, con una proporción aproximada de 80% y 20% y una semilla fija para facilitar la reproducibilidad.

Para evaluar el desempeño se utilizaron Accuracy, AUC y F1-score. Accuracy permite medir la proporción total de predicciones correctas; AUC evalúa la capacidad del modelo para diferenciar entre ambas clases; y F1-score complementa el análisis considerando conjuntamente precisión y recall.

Los resultados obtenidos muestran un buen desempeño del modelo. El Accuracy alcanzó un 83,33%, lo que indica que aproximadamente 8 de cada 10 transacciones fueron clasificadas correctamente. El F1-score también fue de 0,8333, mostrando un buen equilibrio general en la clasificación. Por su parte, el AUC alcanzó 0,8908, siendo el resultado más destacable, ya que refleja una alta capacidad del modelo para diferenciar entre transacciones normales y riesgosas. En conjunto, las métricas muestran que la regresión logística logra capturar de buena manera los patrones presentes en los datos, aunque todavía existe margen para mejorar la clasificación de algunos casos.

Como mejora futura aplicaría validación cruzada y búsqueda de hiperparámetros mediante CrossValidator y ParamGridBuilder. También incorporaría información histórica y variables adicionales que permitan construir una definición de riesgo basada en eventos reales y no solamente en una regla simulada.